In [0]:
import pyspark.sql.functions as f
from pyspark.sql.functions import lit, col
import json

In [0]:
g_env = 'DEV'
g_ucBronze = 'dev_hub_bronze'
v_p_srcSchema = 'lh_ax_idr'
g_ucSilver = 'dev_hub_silver'

##extract using Join

In [0]:
df_tables = spark.sql("""
    SELECT table_catalog, table_schema, table_name
    FROM dev_hub_silver.information_schema.tables
    WHERE table_catalog = 'dev_hub_silver'
    AND
    table_schema = 'lh_ax_idr'
""")

tables = []
for i in df_tables.collect():
    tables.append(i[2])
print(tables)
display(df_tables)

In [0]:
df_meta = spark.table(f'dev_bronze.poc._meta')
display(df_meta)

In [0]:
df_meta = spark.table(f'dev_bronze.poc._meta')
meta_join = df_meta.join(df_tables, f.upper(df_meta.TABLE_NM) == f.upper(df_tables.table_name), 'left')

# _meta_agg_pk = df_meta.groupBy("TABLE_NM").pivot("PK_COL_NM").agg(f.lit(1)).fillna(0)
# display(_meta_agg_pk)
display(meta_join)
table_pk_dict = {}
for row in df_meta.select("TABLE_NM", "PK_COL_NM").collect():
    table = row["TABLE_NM"]
    pk_col = row["PK_COL_NM"]
    if table not in table_pk_dict:
        table_pk_dict[table] = []
    table_pk_dict[table].append(pk_col)

print(table_pk_dict)


In [0]:
dbutils.widgets.get("p_maxWorkers")
p_maxWorkers = 12

In [0]:
import time

##### Function to run function refresh_tbl() in parallel #####
from concurrent.futures import ThreadPoolExecutor

def sync_idr_tables_v2_with_plan(table, pk_cols):
    start_time = time.time()
    sdf = spark.table(f'{g_ucBronze}.{v_p_srcSchema}.{table}')
    print(f"Execution plan for table {table}:")

    sdf.createOrReplaceTempView('sdf')
    pk_cols_str = ', '.join(pk_cols)

    from pyspark.sql import Window
    import pyspark.sql.functions as f

    window_spec = Window.partitionBy(*pk_cols).orderBy(f.col("data_received_utc_dttm").desc())
    deduped = sdf.withColumn("rn", f.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    deduped.write.mode("overwrite").saveAsTable(f"{g_ucSilver}.{v_p_srcSchema}.{table}")

    end_time = time.time()
    print(f'Merge Completed Successfully for {table}')
    print(f'Execution time for {table}: {end_time , start_time:.2f} seconds')







In [0]:
for tbl in table_pk_dict:
    pk_cols = list(table_pk_dict[tbl])
    print(f"Processing table {tbl} with primary key columns: {pk_cols}")


In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback

futures = {}
with ThreadPoolExecutor(max_workers=p_maxWorkers) as ex:
    for tbl in table_pk_dict:
        pk_cols = list(table_pk_dict[tbl])
        futures[ex.submit(sync_idr_tables_v2_with_plan, tbl, pk_cols)] = tbl

errors = []
for fut in as_completed(futures):
    tbl = futures[fut]
    try:
        fut.result()
        print(f"✅ {tbl} done")
    except Exception as e:
        errors.append((tbl, repr(e), traceback.format_exc()))
        print(f"❌ {tbl} failed: {e}")

# only fail the job AFTER everything finished
if errors:
    msg = "\n\n".join([f"TABLE={t}\nERR={err}\n{tb}" for t, err, tb in errors])
    raise Exception("Some tables failed:\n" + msg)